# Interpolation Playground

Use this notebook to quickly run interpolation between any two WAV files in `app/backend/inference/assets`.

Workflow:
1. Put your WAV files in the assets folder.
2. Run cells top-to-bottom once.
3. Choose two files and interpolation settings.
4. Run the generation cell to listen and optionally save output.

In [1]:
from pathlib import Path
import sys

# Resolve project paths from this notebook location
NB_PATH = Path.cwd()
REPO_ROOT = NB_PATH.parents[3] if (NB_PATH / 'Full task.ipynb').exists() else Path('/Users/pol/Documents/UNI/TTM/root/S103-Interface-for-Generative-Audio-Latent-Interpolation')
BACKEND_ROOT = REPO_ROOT / 'app' / 'backend'
ASSETS_DIR = BACKEND_ROOT / 'inference' / 'assets'

if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

print('Repo root:', REPO_ROOT)
print('Assets dir:', ASSETS_DIR)

Repo root: /Users/pol/Documents/UNI/TTM/root/S103-Interface-for-Generative-Audio-Latent-Interpolation
Assets dir: /Users/pol/Documents/UNI/TTM/root/S103-Interface-for-Generative-Audio-Latent-Interpolation/app/backend/inference/assets


In [ ]:
import torch
from io import BytesIO
from IPython.display import Audio, display
import soundfile as sf

from inference.models import AudioElement, InterpolationElement
from inference.methods import AUDIO_ASSET_MAP, get_inference_engine, render_interpolation_audio

In [3]:
# Discover WAV assets
asset_files = sorted(ASSETS_DIR.glob('*.wav'))

if not asset_files:
    print('No WAV files found in assets folder.')
    print('Add files to:', ASSETS_DIR)
else:
    print('Available assets:')
    for i, f in enumerate(asset_files):
        print(f'[{i}] {f.name}')

Available assets:
[0] camp_fire.wav
[1] keyboard.wav


In [5]:
# Load the inference engine once (cached in backend code)
engine = get_inference_engine()
print('Engine loaded. Device:', engine.device)
print('Sample rate:', engine.sr)

Initializing EnCodec processor on device: cpu
Streamable mode: True
  → Enabling streamable mode (disabling chunking)
✓ EnCodec 48kHz model loaded (HuggingFace) - streamable
✓ Sample rate: 48000 Hz
✓ Frame rate: 150 Hz
Engine loaded. Device: cpu
Sample rate: 48000


In [6]:
# ---- Pick sounds and parameters ----
# Use enum-style sound selection (same as backend API)
AUDIO_1 = AudioElement.CAMPFIRE
AUDIO_2 = AudioElement.KEYBOARD

TIMELINE_SIZE = 200
STAY_TIME = 20
STICKYNESS = 3.0
NFE = 32
CONTEXT_STATIC = False

SAVE_OUTPUT = True
OUTPUT_NAME = 'interpolation_output.wav'

In [ ]:
audio_1_path = AUDIO_ASSET_MAP[AUDIO_1]
audio_2_path = AUDIO_ASSET_MAP[AUDIO_2]

if not audio_1_path.exists():
    raise FileNotFoundError(f'Missing audio file: {audio_1_path}')
if not audio_2_path.exists():
    raise FileNotFoundError(f'Missing audio file: {audio_2_path}')

save_path = (ASSETS_DIR / OUTPUT_NAME) if SAVE_OUTPUT else None

print(f'Interpolating: {AUDIO_1.value} -> {AUDIO_2.value}')

interpolation_request = InterpolationElement(
    audio1=AUDIO_1,
    audio2=AUDIO_2,
    timeline_size=TIMELINE_SIZE,
    stay_time=STAY_TIME,
    stickyness=STICKYNESS,
    play=False,
    NFE=NFE,
    context_static=CONTEXT_STATIC,
)

final_audio_bytes = render_interpolation_audio(interpolation_request)

audio_data, sample_rate = sf.read(BytesIO(final_audio_bytes))
display(Audio(audio_data.T if audio_data.ndim > 1 else audio_data, rate=sample_rate))

if save_path:
    save_path.write_bytes(final_audio_bytes)
    print('Saved to:', save_path)